## Importamos librerias

In [2]:
import os
import requests #traer requests de internet
import pandas as pd #traer pandas para trabajar con dataframes
import numpy as np #traer numpy para trabajar con arrays

## Extraccion de data y preparacion

In [3]:
#Configuracion de endpoint de la api y constantes
STANDINGS_URL = "https://race.mx/data/standings.json" #Standings oficiales del chase
VENUES_URL = "https://race.mx/data/venues.json" # Catalago de pistas y coordenadas
CHASE_BASE_POINTS = 2000 # Piso base de puntos en playoffs nascar MX

In [4]:
#Extraccion de datos
print("Descargando datos en vivo de Race.mx...")
try:
    resp_standings = requests.get(STANDINGS_URL, timeout=10).json() #traer datos de la api

    #Convertir a dataframe bajo la lalvi standings
    df_chase = pd.DataFrame(resp_standings["standings"])
    print(f"standing actualizados al: {resp_standings.get('updated', 'N/A')}")

except Exception as e:
    print(f"Error al descargar datos de standings: {e}")

Descargando datos en vivo de Race.mx...
standing actualizados al: 2026-10-18T13:52:04+00:00


## Paso 2: Normalizacion del rendimiento en el campeonato

Bloque 2: Normalización Min-Max Ajustada al Contexto (ChasePerformanceScore)Qué hace: Transforma los puntos crudos del campeonato a una escala relativa entre 0.0 y 1.0.Diferencia clave con Mar Antaya: En prediction1.py y prediction2.py, Mar divide los puntos de cada equipo directamente entre el máximo (p / max_pts) porque en Fórmula 1 los equipos empiezan en 0 puntos. En cambio, en el formato Chase de NASCAR México, los puntos de los contendientes se resetean sobre una base de 2000 puntos.   Si dividiéramos 2020 / 2056, el último lugar tendría un score de 0.982 y el líder 1.000 (casi idénticos, restándole poder predictivo a la variable).Al restar el piso del Chase ($min\_pts = 2000$) mediante la fórmula:

$$\text{ChasePerformanceScore} = \frac{\text{points} - 2000}{\max(\text{points}) - 2000}$$

In [5]:
df_chase["points"].head()

0    2056
1    2051
2    2047
3    2046
4    2046
Name: points, dtype: int64

In [6]:
# En lugar de dividir puntos / max_puntos (como hace Mar en F1 donde empiezan en 0),
# restamos el piso del Chase (2000 pts) para escalar la ventaja real entre 0.0 y 1.0.
max_pts = df_chase["points"].max()
df_chase["ChasePerfomanceScore"] = ((df_chase["points"] - CHASE_BASE_POINTS) / (max_pts - CHASE_BASE_POINTS)).round(4)

## Paso 3: Geolocalizacion automatica dcel autodromo 

In [7]:
try: 
    resp_venues = requests.get(VENUES_URL, timeout=10).json()
    for v in resp_venues.get("venues",[]):
        if "Yucatán" in v.get("name", "") or "Fittipaldi" in v.get("name", ""):
            lat = v.get("latitude") or lat
            lon = v.get("longitude") or lon
            print(f"Sede encontradfa: {v['name']} ({lat}), {lon})")
            break

except Exception as e:
    print(f"Error al descargar datos de venues: {e}")

Sede encontradfa: Autódromo Yucatán (Emerson Fittipaldi) (21.18), -89.65)


## Paso 4: Pronostico meteorologico

In [8]:
API_KEY = os.getenv("OPENWEATHERMAP_API_KEY", "")
weather_url = (f"https://api.openweathermap.org/data/2.5/forecast" f"?lat={lat}&lon={lon}&appid={API_KEY}&units=metric")

In [9]:
try:
    r = requests.get(weather_url, timeout=5)
    data = r.json()

    if str(data.get("cod")) !="200":
        print(f"Aviso de OpenWeather API (cod {data.get('cod')}): {data.get('message')}")

    else:

        # La carrera inicia el 2026-10-18 a las 13:40 CST (UTC-6), que equivale a las 19:40 UTC.
        # Buscamos los bloques trihorarios de OpenWeatherMap que cubren la ventana de carrera:
        slot = next((f for f in data.get("list", []) if f["dt_txt"] in ["2026-10-18 18:00:00", "2026-10-18 21:00:00"]), None)
        if slot:
            # 'pop' = probabilidad de precipitacion
            rain_probability = slot["pop"]
            temperature = slot["main"]["temp"]
            print(f"Clima Openweather (18 Oct): {temperature:.1f}ºC | Prob. de lluvia {rain_probability:.0%}")
        else: 
            print("No se encontraron datos de clima para la fecha de la carrera.")
except Exception as e:
    print(f"error al consultar clima: {e}")

Clima Openweather (18 Oct): 32.0ºC | Prob. de lluvia 45%


## Paso 5: Inspeccion de salida

In [10]:
print("\n-- Vista previa del dataset base ---")
print(df_chase.sort_values("points", ascending=False).to_string(index=False))


-- Vista previa del dataset base ---
 seed                driver  points  ChasePerfomanceScore
    1      Rubén García Jr.    2056                1.0000
    6 Eloy Sebastián Falcón    2051                0.9107
    4          Irwin Vences    2047                0.8393
    3         Rodrigo Rejón    2046                0.8214
    8             Xavi Razo    2046                0.8214
    5         Max Gutiérrez    2042                0.7500
    2      Alex de Alba Jr.    2041                0.7321
    7          Rubén Rovelo    2041                0.7321
    9           Julio Rejón    2041                0.7321
   10     José Luis Ramírez    2021                0.3750


# Fase 2; Feature engineering


## 10 contendientes oficiales del chaces

In [18]:
from sklearn.model_selection import LeaveOneOut
from xgboost import XGBRegressor

In [51]:
df = df_chase.copy()

# Posicion actual por puntos (1 al 10)
df = df.sort_values("points", ascending=False).reset_index(drop=True)
df["CurrentRank"] = df.index + 1

# Gap de puntos contra el lider
leader_pts = df["points"].max()
df["PointsGapToLeader"] = leader_pts - df["points"]

# 3. Momentum en el chase (Lugares ganados o perdidos desde que inicio el playoff)
df["ChaseMomentum"] = df["seed"] - df["CurrentRank"]

# 4. Pilotos destacados en drivers.json (Corregido el error de dedo en el nombre con punto)
featured_in_json = ["Rubén García Jr.", "Alex de Alba Jr."]
df["FeaturedDriver"] = df["driver"].isin(featured_in_json).astype(int)

# 5. Interacción Clima Real (OpenWeather) x posicion en el grupo
df["HeatTrafficStress"] = (
    (temperature / 30.0) * (1 + (df["CurrentRank"] - 1) * 0.05) * (1 + rain_probability)
).round(4)

# 6. Target realista (indice de dominio combinado: 0.0 - favorito a ganar, 1.0 - menos probable)
rank_norm = (df["CurrentRank"] - 1) / 9.0
seed_norm = (df["seed"] - 1) / 9.0
df["TargetDominanceScore"] = (0.65 * rank_norm + 0.35 * seed_norm).round(4)

# FASE 3: ENTRENAMIENTO XGBOOST + LEAVE-ONE-OUT CV

In [52]:
feature_cols = [
    "PointsGapToLeader",     # + brecha de puntos = peor posición (+1)
    "ChasePerfomanceScore",  # + score de puntos = MEJOR posición (-1)
    "ChaseMomentum",         # + remontada en playoffs = MEJOR posición (-1)
    "FeaturedDriver",        # Destacado en drivers.json = MEJOR posición (-1)
    "HeatTrafficStress",     # + desgaste térmico en tráfico = peor posición (+1)
]

X = df[feature_cols].copy()
y = df["TargetDominanceScore"].copy()

In [55]:
# Configuración anti-overfitting para datasets pequeños (N = 10 pilotos)
model_regularizado = XGBRegressor(
    n_estimators=40,        # Menos árboles (evita memorizar las 10 filas)
    learning_rate=0.05,
    max_depth=2,            # Árboles más cortos
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3.0,         # Mayor penalización L2
    monotone_constraints=(1, -1, -1, -1, 1),
    random_state=42,
)

# Validación Leave-One-Out (usando model_regularizado de manera consistente)
loo = LeaveOneOut()
loo_errors = []
for train_idx, test_idx in loo.split(X):
    model_regularizado.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model_regularizado.predict(X.iloc[test_idx])[0]
    loo_errors.append(abs(pred - y.iloc[test_idx].iloc[0]))

# Ajuste final y predicción
model_regularizado.fit(X, y)
df["PredictScore"] = model_regularizado.predict(X).round(4)
final = df.sort_values(["PredictScore", "points"], ascending=[True, False]).reset_index(drop=True)

In [57]:
print("\n" + "=" * 64)
print("   🏁 PREDICCIÓN OFICIAL — GRAN PREMIO YUCATÁN 200")
print("=" * 68)

for i, row in final.iterrows():
    mom = f"+{row["ChaseMomentum"]}" if row["ChaseMomentum"] > 0 else str(row["ChaseMomentum"])
    print(f"    P{i+1:<3} {row['driver']:<24} | Pts: {row['points']} | Momentum: {mom:<3} | Score: {row['PredictScore']:.4f}")

print("\n" + "=" * 68)
print(f"  🥇 P1: {final.iloc[0]['driver']}")
print(f"  🥈 P2: {final.iloc[1]['driver']}")
print(f"  🥉 P3: {final.iloc[2]['driver']}")
print(f"  Leave-One-Out MAE: {np.mean(loo_errors):.4f} | Clima: {temperature:.1f}°C (Lluvia: {rain_probability:.0%})")
print("=" * 68)


   🏁 PREDICCIÓN OFICIAL — GRAN PREMIO YUCATÁN 200
    P1   Rubén García Jr.         | Pts: 2056 | Momentum: 0   | Score: 0.2877
    P2   Eloy Sebastián Falcón    | Pts: 2051 | Momentum: +4  | Score: 0.2913
    P3   Irwin Vences             | Pts: 2047 | Momentum: +1  | Score: 0.3208
    P4   Rodrigo Rejón            | Pts: 2046 | Momentum: -1  | Score: 0.3636
    P5   Xavi Razo                | Pts: 2046 | Momentum: +3  | Score: 0.5045
    P6   Alex de Alba Jr.         | Pts: 2041 | Momentum: -5  | Score: 0.5304
    P7   Max Gutiérrez            | Pts: 2042 | Momentum: -1  | Score: 0.5335
    P8   Rubén Rovelo             | Pts: 2041 | Momentum: -1  | Score: 0.6992
    P9   Julio Rejón              | Pts: 2041 | Momentum: 0   | Score: 0.7185
    P10  José Luis Ramírez        | Pts: 2021 | Momentum: 0   | Score: 0.7226

  🥇 P1: Rubén García Jr.
  🥈 P2: Eloy Sebastián Falcón
  🥉 P3: Irwin Vences
  Leave-One-Out MAE: 0.1980 | Clima: 32.0°C (Lluvia: 45%)


In [32]:
print("\n-- Vista previa del dataset base ---")
print(df_chase.sort_values("points", ascending=False).to_string(index=False))


-- Vista previa del dataset base ---
 seed                driver  points  ChasePerfomanceScore  Temperarure_C  RainProbability
    1      Rubén García Jr.    2056                1.0000          32.05             0.45
    6 Eloy Sebastián Falcón    2051                0.9107          32.05             0.45
    4          Irwin Vences    2047                0.8393          32.05             0.45
    3         Rodrigo Rejón    2046                0.8214          32.05             0.45
    8             Xavi Razo    2046                0.8214          32.05             0.45
    5         Max Gutiérrez    2042                0.7500          32.05             0.45
    2      Alex de Alba Jr.    2041                0.7321          32.05             0.45
    7          Rubén Rovelo    2041                0.7321          32.05             0.45
    9           Julio Rejón    2041                0.7321          32.05             0.45
   10     José Luis Ramírez    2021                0.3750     